Consumer orders, add category and revenue, produce to order-enriched exactly once.

In [2]:
from confluent_kafka import Producer, Consumer

In [3]:
%run order.ipynb

In [4]:
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "order-enricher",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
    'isolation.level': "read_committed",
    "session.timeout.ms": 6000    
})

In [5]:
producer = Producer({
    "bootstrap.servers": "localhost:9092",
    "transactional.id": "order-enricher-1",
    
})
producer.init_transactions()
consumer.subscribe(["orders"])

In [6]:
enriched, idle = 0, 0
while idle < 3:
    batch = consumer.consume(num_messages=100, timeout=1.0)
    if not batch:
        if consumer.assignment():
            idle += 1
            continue
    idle = 0
    producer.begin_transaction()
    for msg in batch:
        if msg.error():
            continue
        try:
            order = decode(msg.value())
        except ValueError as e:
            continue
        if validate(order):
            continue
    order["category"] = PRODUCTS[order["product_id"]][1]
    order["revenue"] = round(order["qty"] * order["price"], 2)
    producer.produce("orders-enriched", key=msg.key(), value=encode(order))
    enriched += 1

    offsets = consumer.position(consumer.assignment())
    producer.send_offsets_to_transaction(offsets, consumer.consumer_group_metadata())
    producer.commit_transaction()
    print(f"transaction commited: {len(batch)} read, {enriched} enriched so far")

consumer.close()
print(f"done: {enriched} enriched orders")

transaction commited: 57 read, 1 enriched so far
done: 1 enriched orders


In [ ]:
# Check the enriched in kafka
# > kafka-console-consumer.sh --bootstrap-server $BS --topic orders-enriched --from-beginning --max-messages 2 --isolation-level read_committed
"""
The consumer rebalance protocol (KIP-848) is production-ready! Set group.protocol=consumer to try it out. See https://kafka.apache.org/documentation/#consumer_rebalance_protocol
{"schema_version": 1, "order_id": "o-9002", "customer_id": "c-010", "product_id": "P300", "qty": 0, "price": 15.0, "city": "Manila", "ts": "2026-11-21T02:20:30", "category": "grocery", "revenue": 0.0}
"""